# Deep Agents: Building Complex Agents for Long-Horizon Tasks

In this notebook, we'll explore **Deep Agents** - a new approach to building AI agents that can handle complex, multi-step tasks over extended periods. We'll implement all four key elements of Deep Agents while building on our Stone Ridge Investment Advisory use case.

**Learning Objectives:**
- Understand the four key elements of Deep Agents: Planning, Context Management, Subagent Spawning, and Long-term Memory
- Implement each element progressively using the `deepagents` package
- Learn to use Skills for progressive capability disclosure
- Use the `deepagents-cli` for interactive agent sessions

## Table of Contents:

- **Breakout Room #1:** Deep Agent Foundations
  - Task 1: Dependencies & Setup
  - Task 2: Understanding Deep Agents
  - Task 3: Planning with Todo Lists
  - Task 4: Context Management with File Systems
  - Task 5: Basic Deep Agent
  - Question #1 & Question #2
  - Activity #1: Build a Research Agent

- **Breakout Room #2:** Advanced Features & Integration
  - Task 6: Subagent Spawning
  - Task 7: Long-term Memory Integration
  - Task 8: Skills - On-Demand Capabilities
  - Task 9: Using deepagents-cli
  - Task 10: Building a Complete Deep Agent System
  - Question #3 & Question #4
  - Activity #2: Build an Investment Advisory Agent

---
# Breakout Room #1
## Deep Agent Foundations

## Task 1: Dependencies & Setup

Before we begin, make sure you have:

1. **API Keys** for:
   - Anthropic (default for Deep Agents) or OpenAI
   - LangSmith (optional, for tracing)
   - Tavily (optional, for web search)

2. **Dependencies installed** via `uv sync`

3. **For the CLI** (Task 9): `uv pip install deepagents-cli`

### Environment Setup

You can either:
- Create a `.env` file with your API keys (recommended):
  ```
  ANTHROPIC_API_KEY=your_key_here
  OPENAI_API_KEY=your_key_here
  LANGCHAIN_API_KEY=your_key_here
  ```
- Or enter them interactively when prompted

In [1]:
# Core imports
import os
import getpass
from uuid import uuid4
from typing import Annotated, TypedDict, Literal

import nest_asyncio
nest_asyncio.apply()  # Required for async operations in Jupyter

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

def get_api_key(env_var: str, prompt: str) -> str:
    """Get API key from environment or prompt user."""
    value = os.environ.get(env_var, "")
    if not value:
        value = getpass.getpass(prompt)
        if value:
            os.environ[env_var] = value
    return value

In [2]:
# Set Anthropic API Key (default for Deep Agents)
anthropic_key = get_api_key("ANTHROPIC_API_KEY", "Anthropic API Key: ")
if anthropic_key:
    print("Anthropic API key set")
else:
    print("Warning: No Anthropic API key configured")

Anthropic API key set


In [3]:
# Optional: OpenAI for alternative models and subagents
openai_key = get_api_key("OPENAI_API_KEY", "OpenAI API Key (press Enter to skip): ")
if openai_key:
    print("OpenAI API key set")
else:
    print("OpenAI API key not configured (optional)")

OpenAI API key set


In [4]:
# Optional: LangSmith for tracing
langsmith_key = get_api_key("LANGCHAIN_API_KEY", "LangSmith API Key (press Enter to skip): ")

if langsmith_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = f"AIE9 - Deep Agents - {uuid4().hex[0:8]}"
    print(f"LangSmith tracing enabled. Project: {os.environ['LANGCHAIN_PROJECT']}")
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("LangSmith tracing disabled")

LangSmith tracing enabled. Project: AIE9 - Deep Agents - f9910ac8


In [5]:
# Verify deepagents installation
from deepagents import create_deep_agent
print("deepagents package imported successfully!")

# Test with a simple agent
test_agent = create_deep_agent()
result = test_agent.invoke({
    "messages": [{"role": "user", "content": "Say 'Deep Agents ready!' in exactly those words."}]
})
print(result["messages"][-1].content)

deepagents package imported successfully!
Deep Agents ready!


## Task 2: Understanding Deep Agents

**Deep Agents** represent a shift from simple tool-calling loops to sophisticated agents that can handle complex, long-horizon tasks. They address four key challenges:

### The Four Key Elements

| Element | Challenge Addressed | Implementation |
|---------|---------------------|----------------|
| **Planning** | "What should I do?" | Todo lists that persist task state |
| **Context Management** | "What do I know?" | File systems for storing/retrieving info |
| **Subagent Spawning** | "Who can help?" | Task tool for delegating to specialists |
| **Long-term Memory** | "What did I learn?" | LangGraph Store for cross-session memory |

### Deep Agents vs Traditional Agents

```
Traditional Agent Loop:
\u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510
\u2502  User Query                         \u2502
\u2502       \u2193                             \u2502
\u2502  Think \u2192 Act \u2192 Observe \u2192 Repeat     \u2502
\u2502       \u2193                             \u2502
\u2502  Response                           \u2502
\u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518
Problems: Context bloat, no delegation,
          loses track of complex tasks

Deep Agent Architecture:
\u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510
\u2502                    Deep Agent                           \u2502
\u251c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2524
\u2502  \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510  \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510  \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510   \u2502
\u2502  \u2502   PLANNING   \u2502  \u2502   CONTEXT    \u2502  \u2502   MEMORY     \u2502   \u2502
\u2502  \u2502              \u2502  \u2502  MANAGEMENT  \u2502  \u2502              \u2502   \u2502
\u2502  \u2502 write_todos  \u2502  \u2502              \u2502  \u2502   Store      \u2502   \u2502
\u2502  \u2502 update_todo  \u2502  \u2502  read_file   \u2502  \u2502  namespace   \u2502   \u2502
\u2502  \u2502 list_todos   \u2502  \u2502  write_file  \u2502  \u2502  get/put     \u2502   \u2502
\u2502  \u2502              \u2502  \u2502  edit_file   \u2502  \u2502              \u2502   \u2502
\u2502  \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518  \u2502  ls          \u2502  \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518   \u2502
\u2502                    \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518                     \u2502
\u2502  \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510   \u2502
\u2502  \u2502              SUBAGENT SPAWNING                   \u2502   \u2502
\u2502  \u2502                                                  \u2502   \u2502
\u2502  \u2502  task(prompt, tools, model, system_prompt)       \u2502   \u2502
\u2502  \u2502       \u2193              \u2193              \u2193            \u2502   \u2502
\u2502  \u2502  \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510    \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510    \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510          \u2502   \u2502
\u2502  \u2502  \u2502Research\u2502    \u2502Writing \u2502    \u2502Analysis\u2502          \u2502   \u2502
\u2502  \u2502  \u2502Subagent\u2502    \u2502Subagent\u2502    \u2502Subagent\u2502          \u2502   \u2502
\u2502  \u2502  \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518    \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518    \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518          \u2502   \u2502
\u2502  \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518   \u2502
\u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518
```

### When to Use Deep Agents

| Use Case | Traditional Agent | Deep Agent |
|----------|-------------------|------------|
| Simple Q&A | Yes | Overkill |
| Single-step tool use | Yes | Overkill |
| Multi-step research | May lose track | Yes |
| Complex projects | Context overflow | Yes |
| Parallel task execution | No | Yes |
| Long-running sessions | No | Yes |

### Key Insight: "Planning is Context Engineering"

Deep Agents treat planning not as a separate phase, but as **context engineering**:
- Todo lists aren't just task trackers--they're **persistent context** about what to do
- File systems aren't just storage--they're **extended memory** beyond the context window
- Subagents aren't just helpers--they're **context isolation** to prevent bloat

## Task 3: Planning with Todo Lists

The first key element of Deep Agents is **Planning**. Instead of trying to hold all task state in the conversation, Deep Agents use structured todo lists.

### Why Todo Lists?

1. **Persistence**: Tasks survive across conversation turns
2. **Visibility**: Both agent and user can see progress
3. **Structure**: Clear tracking of what's done vs pending
4. **Recovery**: Agent can resume from where it left off

### Todo List Tools

| Tool | Purpose |
|------|----------|
| `write_todos` | Create a structured task list |
| `update_todo` | Mark tasks as complete/in-progress |
| `list_todos` | View current task state |

In [6]:
from langchain_core.tools import tool
from typing import List, Optional
import json

# Simple in-memory todo storage for demonstration
# In production, Deep Agents use persistent storage
TODO_STORE = {}

@tool
def write_todos(todos: List[dict]) -> str:
    """Create a list of todos for tracking task progress.
    
    Args:
        todos: List of todo items, each with 'title' and optional 'description'
    
    Returns:
        Confirmation message with todo IDs
    """
    created = []
    for i, todo in enumerate(todos):
        todo_id = f"todo_{len(TODO_STORE) + i + 1}"
        TODO_STORE[todo_id] = {
            "id": todo_id,
            "title": todo.get("title", "Untitled"),
            "description": todo.get("description", ""),
            "status": "pending"
        }
        created.append(todo_id)
    return f"Created {len(created)} todos: {', '.join(created)}"

@tool
def update_todo(todo_id: str, status: Literal["pending", "in_progress", "completed"]) -> str:
    """Update the status of a todo item.
    
    Args:
        todo_id: The ID of the todo to update
        status: New status (pending, in_progress, completed)
    
    Returns:
        Confirmation message
    """
    if todo_id not in TODO_STORE:
        return f"Todo {todo_id} not found"
    TODO_STORE[todo_id]["status"] = status
    return f"Updated {todo_id} to {status}"

@tool
def list_todos() -> str:
    """List all todos with their current status.
    
    Returns:
        Formatted list of all todos
    """
    if not TODO_STORE:
        return "No todos found"
    
    result = []
    for todo_id, todo in TODO_STORE.items():
        status_emoji = {"pending": "[ ]", "in_progress": "[~]", "completed": "[x]"}
        emoji = status_emoji.get(todo["status"], "[?]")
        result.append(f"{emoji} [{todo_id}] {todo['title']} ({todo['status']})")
    return "\n".join(result)

print("Todo tools defined!")

Todo tools defined!


In [7]:
# Test the todo tools
TODO_STORE.clear()  # Reset for demo

# Create some investment advisory todos
result = write_todos.invoke({
    "todos": [
        {"title": "Assess current portfolio allocation", "description": "Review user's current holdings and asset mix"},
        {"title": "Research alternative investment opportunities", "description": "Find evidence-based strategies"},
        {"title": "Create personalized investment strategy", "description": "Combine findings into actionable steps"},
    ]
})
print(result)
print("\nCurrent todos:")
print(list_todos.invoke({}))

Created 3 todos: todo_1, todo_3, todo_5

Current todos:
[ ] [todo_1] Assess current portfolio allocation (pending)
[ ] [todo_3] Research alternative investment opportunities (pending)
[ ] [todo_5] Create personalized investment strategy (pending)


In [8]:
# Simulate progress
update_todo.invoke({"todo_id": "todo_1", "status": "completed"})
update_todo.invoke({"todo_id": "todo_2", "status": "in_progress"})

print("After updates:")
print(list_todos.invoke({}))

After updates:
[x] [todo_1] Assess current portfolio allocation (completed)
[ ] [todo_3] Research alternative investment opportunities (pending)
[ ] [todo_5] Create personalized investment strategy (pending)


## Task 4: Context Management with File Systems

The second key element is **Context Management**. Deep Agents use file systems to:

1. **Offload large content** - Store research, documents, and results to disk
2. **Persist across sessions** - Files survive beyond conversation context
3. **Share between subagents** - Subagents can read/write shared files
4. **Prevent context overflow** - Large tool results automatically saved to disk

### Automatic Context Management

Deep Agents automatically handle context limits:
- **Large result offloading**: Tool results >20k tokens -> saved to disk
- **Proactive offloading**: At 85% context capacity -> agent saves state to disk
- **Summarization**: Long conversations get summarized while preserving intent

### File System Tools

| Tool | Purpose |
|------|----------|
| `ls` | List directory contents |
| `read_file` | Read file contents |
| `write_file` | Create/overwrite files |
| `edit_file` | Make targeted edits |

In [9]:
import os
from pathlib import Path

# Create a workspace directory for our agent
WORKSPACE = Path("workspace")
WORKSPACE.mkdir(exist_ok=True)

@tool
def ls(path: str = ".") -> str:
    """List contents of a directory.
    
    Args:
        path: Directory path to list (default: current directory)
    
    Returns:
        List of files and directories
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"Directory not found: {path}"
    
    items = []
    for item in sorted(target.iterdir()):
        prefix = "[DIR]" if item.is_dir() else "[FILE]"
        size = f" ({item.stat().st_size} bytes)" if item.is_file() else ""
        items.append(f"{prefix} {item.name}{size}")
    
    return "\n".join(items) if items else "(empty directory)"

@tool
def read_file(path: str) -> str:
    """Read contents of a file.
    
    Args:
        path: Path to the file to read
    
    Returns:
        File contents
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    return target.read_text()

@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file (creates or overwrites).
    
    Args:
        path: Path to the file to write
        content: Content to write to the file
    
    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return f"Wrote {len(content)} characters to {path}"

@tool
def edit_file(path: str, old_text: str, new_text: str) -> str:
    """Edit a file by replacing text.
    
    Args:
        path: Path to the file to edit
        old_text: Text to find and replace
        new_text: Replacement text
    
    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    
    content = target.read_text()
    if old_text not in content:
        return f"Text not found in {path}"
    
    new_content = content.replace(old_text, new_text, 1)
    target.write_text(new_content)
    return f"Updated {path}"

print("File system tools defined!")
print(f"Workspace: {WORKSPACE.absolute()}")

File system tools defined!
Workspace: /Users/alexei.naumann/Desktop/AIE/AI-Engineering/08_Advanced_Retrieval_and_Deep_Research/workspace


In [10]:
# Test the file system tools
print("Current workspace contents:")
print(ls.invoke({"path": "."}))

Current workspace contents:
[FILE] .gitkeep (0 bytes)


In [11]:
# Create a research notes file
notes = """# Alternative Investments Research Notes

## Key Findings
- Alternatives can improve portfolio diversification
- Reinsurance offers uncorrelated returns
- Private equity requires longer time horizons

## TODO
- [ ] Review individual client needs
- [ ] Create personalized recommendations
"""

result = write_file.invoke({"path": "research/investment_notes.md", "content": notes})
print(result)

# Verify it was created
print("\nResearch directory:")
print(ls.invoke({"path": "research"}))

Wrote 288 characters to research/investment_notes.md

Research directory:
[FILE] investment_notes.md (288 bytes)


In [12]:
# Read and edit the file
print("File contents:")
print(read_file.invoke({"path": "research/investment_notes.md"}))

File contents:
# Alternative Investments Research Notes

## Key Findings
- Alternatives can improve portfolio diversification
- Reinsurance offers uncorrelated returns
- Private equity requires longer time horizons

## TODO
- [ ] Review individual client needs
- [ ] Create personalized recommendations



## Task 5: Basic Deep Agent

Now let's create a basic Deep Agent using the `deepagents` package. This combines:
- Planning (todo lists)
- Context management (file system)
- A capable LLM backbone

### Configuring the FilesystemBackend

Deep Agents come with **built-in file tools** (`ls`, `read_file`, `write_file`, `edit_file`). To control where files are stored, we configure a `FilesystemBackend`:

```python
from deepagents.backends import FilesystemBackend

backend = FilesystemBackend(
    root_dir="/path/to/workspace",
    virtual_mode=True  # REQUIRED to actually sandbox files!
)
```

**Critical: `virtual_mode=True`**
- Without `virtual_mode=True`, agents can still write anywhere on the filesystem!
- The `root_dir` alone does NOT restrict file access
- `virtual_mode=True` blocks paths with `..`, `~`, and absolute paths outside root

In [13]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Configure the filesystem backend to use our workspace directory
# IMPORTANT: virtual_mode=True is required to actually restrict paths to root_dir
# Without it, agents can still write anywhere on the filesystem!
workspace_path = Path("workspace").absolute()
filesystem_backend = FilesystemBackend(
    root_dir=str(workspace_path),
    virtual_mode=True  # This is required to sandbox file operations!
)

# Combine our custom tools (for todo tracking)
# Note: Deep Agents has built-in file tools (ls, read_file, write_file, edit_file)
# that will use the configured FilesystemBackend
custom_tools = [
    write_todos,
    update_todo,
    list_todos,
]

# Create a basic Deep Agent
investment_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=custom_tools,
    backend=filesystem_backend,  # Configure where files are stored
    system_prompt="""You are a Stone Ridge Investment Advisory Assistant that helps users make informed investment decisions.

When given a complex task:
1. First, create a todo list to track your progress
2. Work through each task, updating status as you go
3. Save important findings to files for reference
4. Provide a clear summary when complete

Be thorough but concise. Always explain your reasoning."""
)

print(f"Basic Deep Agent created!")
print(f"File operations sandboxed to: {workspace_path}")

Basic Deep Agent created!
File operations sandboxed to: /Users/alexei.naumann/Desktop/AIE/AI-Engineering/08_Advanced_Retrieval_and_Deep_Research/workspace


In [15]:
# Reset todo store for fresh demo
TODO_STORE.clear()

# Test with a multi-step investment advisory task
result = investment_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """I want to diversify my portfolio into alternative investments. I currently have:
- 60% equities, 30% bonds, 10% cash
- No exposure to alternatives
- Limited knowledge of reinsurance and private equity.

Please create a personalized alternative investment strategy and save it to a file."""
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
## Summary

I've created a comprehensive personalized alternative investment strategy saved to `/personalized_alternative_investment_strategy.md`. The strategy addresses your current 60/30/10 allocation and provides a practical roadmap for diversification.

**Key Highlights:**

**Phase 1 (Months 1-6): 15% Alternatives**
- Start with liquid, accessible options: Public REITs (VNQ), Commodities (DBC), Liquid Alternatives (AQRIX), Infrastructure (MLPX)
- Low minimums ($50-$1,000) and daily liquidity

**Phase 2 (Months 7-12): 25% Alternatives**
- Add private REITs via Fundrise/YieldStreet
- Include gold and selective cryptocurrency exposure
- Higher returns but reduced liquidity

**Phase 3 (Optional): 30-35% Alternatives**
- Advanced strategies like private equity through BDCs/interval funds
- Reinsurance exposure for ultimate diversification
- Higher minimums and complexity

**Target Final Allocation:**
- 45% Equities (↓15%), 20% Bonds (↓10%), 5% Cash (↓5%), 30% Alternative

In [16]:
# Check what the agent created
print("Todo list after task:")
print(list_todos.invoke({}))

print("\n" + "="*50)
print("\nWorkspace contents:")
# List files in the workspace directory
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    else:
        print(f"  [DIR] {f.name}/")

Todo list after task:
[x] [todo_1] Analyze current portfolio composition (completed)
[x] [todo_3] Research reinsurance investment opportunities (completed)
[x] [todo_5] Research private equity investment options (completed)
[x] [todo_7] Evaluate other alternative investment categories (completed)
[x] [todo_9] Develop portfolio allocation recommendations (completed)
[x] [todo_11] Create implementation strategy (completed)
[x] [todo_13] Compile comprehensive strategy document (completed)
[x] [todo_8] Research REITs (Real Estate Investment Trusts) (completed)
[x] [todo_10] Research Commodities (completed)
[x] [todo_12] Research Hedge Funds and Liquid Alternatives (completed)
[x] [todo_14] Research Infrastructure Investments (completed)
[x] [todo_16] Research fundamentals of reinsurance investments (completed)
[x] [todo_18] Research Cryptocurrency and Digital Assets (completed)
[x] [todo_20] Create Comprehensive Analysis Document (completed)
[x] [todo_22] Rank Categories by Suitability (co

---
## Question #1:

What are the **trade-offs** of using todo lists for planning? Consider:
- When might explicit planning overhead slow things down?
- How granular should todo items be?
- What happens if the agent creates todos but never completes them?

##### Answer:
ToDo lists are useful for adding structure and observability to an LLM inference process that is inherently difficult to observe and reason about. They provide a sequence of tasks for the agent to accomplish in a structured way that can make the information retrieval process simpler and self-adjusting.<br><br>
ToDo lists may slow things down if they are being used to generate an answer to a simple one-part question that only needed to use one tool for getting a response. If the ToDo items are too general, the agent may initiate a process that is not repeatable in the future or it may take too long to interpret the actual ToDo task. If the ToDo items are too specific, it may cause the agent to spend too much time on a single task that is not related to the overall agent goal (like counting specific words in a paragraph, when the ask was just to determine the meaning of a single sentence). If ToDo items are created but never completed, the structure enabled by the list becomes less useful and valuable compute resources (and context space) may be spent on items that are never actually utilized in getting the final answer. 

## Question #2:

How would you design a **context management strategy** for an investment advisory agent that:
- Needs to reference a large investments handbook (16KB)
- Tracks client metrics over time
- Must remember client constraints (risk tolerance, regulatory status) for safety

What goes in files vs. in the prompt? What should never be offloaded?

##### Answer:
Anything that is critical for making financial calculations, tracking regulatory status or evaluating risk criteria should always be stored in the prompt, for the fastest and most deterministic level of access. Things like historical client metrics can be moved to external storage and accessed only when needed. Semantic chunking can be used to break up the large stored text objects and provide faster retrievals for specific facets of information. If the context gets bloated, you can keep a rolling condensed summary of the context between turns and compact that context every few turns. 

---
## Activity #1: Build a Research Agent

Build a Deep Agent that can research an investment topic and produce a structured report.

### Requirements:
1. Create todos for the research process
2. Read from the AlternativeInvestmentsHandbook.txt in the data folder
3. Save findings to a structured markdown file
4. Update todo status as tasks complete

### Test prompt:
"Research alternative investment strategies and create a comprehensive guide with at least 5 evidence-based approaches."

In [17]:
### YOUR CODE HERE ###

# Step 1 & 2: Create a tool to read the investments handbook from the data folder
from pathlib import Path

@tool
def read_handbook(section: Optional[str] = None) -> str:
    """Read the Alternative Investments Handbook from the data folder.

    Args:
        section: Optional chapter or section keyword to search for.
                 If None, returns the full handbook.

    Returns:
        The handbook text (or the matching section).
    """
    handbook_path = Path("data/AlternativeInvestmentsHandbook.txt")
    if not handbook_path.exists():
        return "Handbook not found at data/AlternativeInvestmentsHandbook.txt"
    content = handbook_path.read_text()
    if section:
        # Return paragraphs that mention the keyword (case-insensitive)
        paragraphs = content.split("\n\n")
        matches = [p for p in paragraphs if section.lower() in p.lower()]
        if matches:
            return "\n\n".join(matches)
        return f"No section matching '{section}' found in the handbook."
    return content

# Step 3: Create the research agent with a research-focused system prompt
research_tools = [
    write_todos,
    update_todo,
    list_todos,
    read_handbook,
]

research_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=research_tools,
    backend=filesystem_backend,
    system_prompt="""You are a senior investment research analyst at Stone Ridge.

Your workflow for any research task:
1. Create a structured todo list that breaks the research into clear steps.
2. Read the Alternative Investments Handbook using the read_handbook tool to gather evidence.
3. Synthesize findings into a well-structured markdown report.
4. Save the report to a file using write_file.
5. Update each todo to 'completed' as you finish it.

Always cite specific data, percentages, and examples from the handbook.
Be thorough, evidence-based, and actionable.""",
)

# Step 4: Test with the alternative investment research task
TODO_STORE.clear()

result = research_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Research alternative investment strategies and create a comprehensive "
            "guide with at least 5 evidence-based approaches. Read the handbook for "
            "supporting data and save your final report to "
            "'research/alternative_strategies_guide.md'."
        ),
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I've successfully created a comprehensive alternative investment strategies guide based on extensive evidence from the Alternative Investments Handbook. The guide covers seven distinct strategies:

## Key Strategies Analyzed:

1. **Real Estate (REITs/Private)** - Income generation, inflation protection, moderate diversification
2. **Insurance-Linked Securities** - Zero market correlation, 5-8% spreads, crisis protection  
3. **Managed Futures** - Trend-following, crisis performance, -0.1 to 0.2 equity correlation
4. **Private Equity/Venture Capital** - Illiquidity premiums, 2-3x returns, J-curve patterns
5. **Infrastructure** - 4-8% yields, inflation linkage, essential services stability
6. **Gold/Precious Metals** - Store of value, 25% gain in 2008 crisis, zero correlation
7. **Hedge Fund Strategies** - Risk-adjusted returns, professional management, diversified approaches

## Evidence-Based Features:
- **Specific Performance Data**: Historical returns, correlat

---
# Breakout Room #2
## Advanced Features & Integration

## Task 6: Subagent Spawning

The third key element is **Subagent Spawning**. This allows a Deep Agent to delegate tasks to specialized subagents.

### Why Subagents?

1. **Context Isolation**: Each subagent has its own context window, preventing bloat
2. **Specialization**: Different subagents can have different tools/prompts
3. **Parallelism**: Multiple subagents can work simultaneously
4. **Cost Optimization**: Use cheaper models for simpler subtasks

### How Subagents Work

```
Main Agent
    |-- task("Research reinsurance strategies", model="gpt-4o-mini")
    |       |-- Returns: Summary of findings
    |
    |-- task("Analyze portfolio allocation data", tools=[analyze_tool])
    |       |-- Returns: Analysis results
    |
    |-- task("Write investment recommendations", system_prompt="Be concise")
            |-- Returns: Final recommendations
```

Key benefit: The main agent only receives **summaries**, not all the intermediate context!

In [18]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Define specialized subagent configurations
# Note: Subagents inherit the backend from the parent agent
research_subagent = {
    "name": "research-agent",
    "description": "Use this agent to research investment topics in depth. It can read documents and synthesize information.",
    "system_prompt": """You are an investment research specialist. Your job is to:
1. Find relevant information in provided documents
2. Synthesize findings into clear summaries
3. Cite sources when possible

Be thorough but concise. Focus on evidence-based information.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",  # Cheaper model for research
}

writing_subagent = {
    "name": "writing-agent",
    "description": "Use this agent to create well-structured documents, plans, and guides.",
    "system_prompt": """You are an investment content writer. Your job is to:
1. Take research findings and turn them into clear, actionable content
2. Structure information for easy understanding
3. Use formatting (headers, bullets, etc.) effectively

Write in a professional, analytical tone.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "anthropic:claude-sonnet-4-20250514",
}

print("Subagent configurations defined!")

Subagent configurations defined!


In [19]:
# Create a coordinator agent that can spawn subagents
coordinator_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[write_todos, update_todo, list_todos],
    backend=filesystem_backend,  # Use the same backend - subagents inherit it
    subagents=[research_subagent, writing_subagent],
    system_prompt="""You are an Investment Advisory Coordinator. Your role is to:
1. Break down complex investment requests into subtasks
2. Delegate research to the research-agent
3. Delegate content creation to the writing-agent
4. Coordinate the overall workflow using todos

Use subagents for specialized work rather than doing everything yourself.
This keeps the work organized and the results high-quality."""
)

print("Coordinator agent created with subagent capabilities!")

Coordinator agent created with subagent capabilities!


In [20]:
# Reset for demo
TODO_STORE.clear()

# Test the coordinator with a complex task
result = coordinator_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Create a comprehensive quarterly investment review framework.

The review should:
1. Research current market conditions for alternatives
2. Include analysis for reinsurance, private equity, and real estate
3. Be saved as a well-formatted markdown file"""
    }]
})

print("Coordinator response:")
print(result["messages"][-1].content)

Coordinator response:
Perfect! I've successfully created a comprehensive quarterly investment review framework that incorporates current market research for alternative investments. Here's what has been delivered:

## Summary of Completed Work:

### ✅ **Market Research Completed**
- **Reinsurance Markets**: Current pricing showing 5-10% increases, improving capacity, rate hardening expected to continue
- **Private Equity**: Moderate deal flow decline, stable valuations at 10-12x EBITDA, strong interest in tech/healthcare
- **Real Estate**: Residential strength vs. commercial headwinds, interest rate impacts, multi-family and industrial opportunities

### ✅ **Comprehensive Framework Created**
The framework document includes:

**1. Executive Summary Section** - Performance summaries and key recommendations  
**2. Market Overview** - Macroeconomic analysis and market performance review  
**3. Alternative Investments Analysis** - Detailed sections for:
   - Reinsurance portfolio analysis w

In [21]:
# Check the results
print("Final todo status:")
print(list_todos.invoke({}))

print("\nGenerated files in workspace:")
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

Final todo status:
[x] [todo_1] Research current market conditions for alternative investments (completed)
[x] [todo_3] Create comprehensive quarterly investment review framework (completed)

Generated files in workspace:
  [FILE] .gitkeep (0 bytes)
  [FILE] alternative_investments_analysis.md (21366 bytes)
  [FILE] alternative_investments_research.md (21592 bytes)
  [FILE] comprehensive_quarterly_investment_review_framework.md (23829 bytes)
  [DIR] market_research/
  [FILE] personalized_alternative_investment_strategy.md (9248 bytes)
  [FILE] quarterly_investment_review_framework.md (29416 bytes)
  [FILE] reinsurance_investment_analysis.md (13461 bytes)
  [DIR] research/


## Task 7: Long-term Memory Integration

The fourth key element is **Long-term Memory**. Deep Agents integrate with LangGraph's Store for persistent memory across sessions.

### Memory Types in Deep Agents

| Type | Scope | Use Case |
|------|-------|----------|
| **Thread Memory** | Single conversation | Current session context |
| **User Memory** | Across threads, per user | User preferences, history |
| **Shared Memory** | Across all users | Common knowledge, learned patterns |

### Integration with LangGraph Store

Deep Agents can use the same `InMemoryStore` (or `PostgresStore`) we learned in Session 6:

In [22]:
from langgraph.store.memory import InMemoryStore

# Create a memory store
memory_store = InMemoryStore()

# Store user profile
user_id = "user_alex"
profile_namespace = (user_id, "profile")

memory_store.put(profile_namespace, "name", {"value": "Alex"})
memory_store.put(profile_namespace, "goals", {
    "primary": "maximize risk-adjusted returns",
    "secondary": "diversify into alternatives"
})
memory_store.put(profile_namespace, "conditions", {
    "constraints": ["ESG-compliant"],
    "regulatory": ["accredited investor"]
})
memory_store.put(profile_namespace, "preferences", {
    "review_frequency": "quarterly",
    "communication_style": "detailed"
})

print(f"Stored profile for {user_id}")

# Retrieve and display
for item in memory_store.search(profile_namespace):
    print(f"  {item.key}: {item.value}")

Stored profile for user_alex
  name: {'value': 'Alex'}
  goals: {'primary': 'maximize risk-adjusted returns', 'secondary': 'diversify into alternatives'}
  conditions: {'constraints': ['ESG-compliant'], 'regulatory': ['accredited investor']}
  preferences: {'review_frequency': 'quarterly', 'communication_style': 'detailed'}


In [23]:
# Create memory-aware tools
from langgraph.store.base import BaseStore

@tool
def get_user_profile(user_id: str) -> str:
    """Retrieve a user's investment profile from long-term memory.
    
    Args:
        user_id: The user's unique identifier
    
    Returns:
        User profile as formatted text
    """
    namespace = (user_id, "profile")
    items = list(memory_store.search(namespace))
    
    if not items:
        return f"No profile found for {user_id}"
    
    result = [f"Profile for {user_id}:"]
    for item in items:
        result.append(f"  {item.key}: {item.value}")
    return "\n".join(result)

@tool
def save_user_preference(user_id: str, key: str, value: str) -> str:
    """Save a user preference to long-term memory.
    
    Args:
        user_id: The user's unique identifier
        key: The preference key
        value: The preference value
    
    Returns:
        Confirmation message
    """
    namespace = (user_id, "preferences")
    memory_store.put(namespace, key, {"value": value})
    return f"Saved preference '{key}' for {user_id}"

print("Memory tools defined!")

Memory tools defined!


In [24]:
# Create a memory-enhanced agent
memory_tools = [
    get_user_profile,
    save_user_preference,
    write_todos,
    update_todo,
    list_todos,
]

memory_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=memory_tools,
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a Stone Ridge Investment Advisory Assistant with long-term memory.

At the start of each conversation:
1. Check the user's profile to understand their goals and constraints
2. Personalize all advice based on their profile
3. Save any new preferences they mention

Always reference stored information to show you remember the user."""
)

print("Memory-enhanced agent created!")

Memory-enhanced agent created!


In [25]:
# Test the memory agent
TODO_STORE.clear()

result = memory_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Hi! My user_id is user_alex. What alternative investment allocation would you recommend for me?"
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
Based on your profile, Alex, I can see you're focused on maximizing risk-adjusted returns while diversifying into alternatives, with ESG compliance as a key constraint. As an accredited investor, you have access to the full range of alternative investments.

Here's my recommended alternative investment allocation for you:

**Core Alternative Allocation (15-25% of total portfolio):**

1. **Real Estate (8-12%)**
   - ESG-focused REITs and real estate funds
   - Opportunity zones for tax-efficient growth
   - Consider sustainable/green building-focused funds

2. **Private Credit (3-5%)**
   - Direct lending funds with ESG screening
   - Focus on funds supporting sustainable business practices
   - Provides steady income with inflation protection

3. **Hedge Fund Strategies (2-4%)**
   - Long/short equity funds with ESG integration
   - Market-neutral strategies for downside protection
   - Avoid funds with significant fossil fuel exposure

4. **Commodities/Infrastructure (

## Task 8: Skills - On-Demand Capabilities

**Skills** are a powerful feature for progressive capability disclosure. Instead of loading all tools upfront, agents can load specialized capabilities on demand.

### Why Skills?

1. **Context Efficiency**: Don't waste context on unused tool descriptions
2. **Specialization**: Skills can include detailed instructions for specific tasks
3. **Modularity**: Easy to add/remove capabilities
4. **Discoverability**: Agent can browse available skills

### SKILL.md Format

Skills are defined in markdown files with YAML frontmatter:

```markdown
---
name: skill-name
description: What this skill does
version: 1.0.0
tools:
  - tool1
  - tool2
---

# Skill Instructions

Detailed steps for how to use this skill...
```

In [26]:
# Let's look at the skills we created
skills_dir = Path("skills")

print("Available skills:")
for skill_dir in skills_dir.iterdir():
    if skill_dir.is_dir():
        skill_file = skill_dir / "SKILL.md"
        if skill_file.exists():
            content = skill_file.read_text()
            # Extract name and description from frontmatter
            lines = content.split("\n")
            name = ""
            desc = ""
            for line in lines:
                if line.startswith("name:"):
                    name = line.split(":", 1)[1].strip()
                if line.startswith("description:"):
                    desc = line.split(":", 1)[1].strip()
            print(f"  - {name}: {desc}")

Available skills:
  - investment-research: Research investment opportunities and create comparison reports
  - portfolio-assessment: Assess client investment portfolio and create personalized recommendations


In [27]:
# Read the portfolio-assessment skill
skill_content = Path("skills/portfolio-assessment/SKILL.md").read_text()
print(skill_content)

---
name: portfolio-assessment
description: Assess client investment portfolio and create personalized recommendations
version: 1.0.0
tools:
  - read_file
  - write_file
---

# Portfolio Assessment Skill

You are conducting a comprehensive investment portfolio assessment. Follow these steps:

## Step 1: Gather Information
Ask the client about:
- Current portfolio allocation (equities, bonds, alternatives, cash)
- Investment goals (growth, income, preservation, diversification)
- Risk tolerance and investment horizon
- Regulatory status (accredited investor, qualified purchaser, etc.)
- Any constraints or preferences (ESG, sector exclusions, liquidity needs)
- Current knowledge of alternative investments

## Step 2: Analyze Responses
Review the client's answers and identify:
- Primary investment priority
- Secondary goals
- Potential barriers to implementation (liquidity, minimums, accreditation)
- Existing portfolio strengths to build on

## Step 3: Create Assessment Report
Write a por

In [28]:
# Create a skill-aware tool
@tool
def load_skill(skill_name: str) -> str:
    """Load a skill's instructions for a specialized task.
    
    Available skills:
    - portfolio-assessment: Assess client portfolio and create recommendations
    - investment-research: Research investment opportunities and strategies
    
    Args:
        skill_name: Name of the skill to load
    
    Returns:
        Skill instructions
    """
    skill_path = Path(f"skills/{skill_name}/SKILL.md")
    if not skill_path.exists():
        available = [d.name for d in Path("skills").iterdir() if d.is_dir()]
        return f"Skill '{skill_name}' not found. Available: {', '.join(available)}"
    
    return skill_path.read_text()

print("Skill loader defined!")

Skill loader defined!


In [29]:
# Create an agent that can load and use skills
skill_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        load_skill,
        write_todos,
        update_todo,
        list_todos,
    ],
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are an investment advisory assistant with access to specialized skills.

When a user asks for something that matches a skill:
1. Load the appropriate skill using load_skill()
2. Follow the skill's instructions carefully
3. Save outputs as specified in the skill

Available skills:
- portfolio-assessment: For comprehensive portfolio evaluations
- investment-research: For researching investment opportunities and strategies

If no skill matches, use your general investment knowledge."""
)

print("Skill-aware agent created!")

Skill-aware agent created!


In [30]:
# Test with a skill-appropriate request
TODO_STORE.clear()

result = skill_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "I'd like a portfolio assessment. I'm a 35-year-old tech executive with a $2M portfolio, moderate risk tolerance, and want to allocate 20% to alternatives. I'm an accredited investor with no current alternative exposure."
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
## Portfolio Assessment Complete

Based on your profile, here are my recommendations organized by implementation timeline:

### Immediate Action Items (This Quarter)
1. **Establish Alternative Investment Research Process**
   - Create due diligence framework for evaluating alternative managers
   - Begin researching private equity, real estate, and hedge fund options
   - Set up meetings with 3-5 alternative investment platforms

2. **Conduct Portfolio Risk Assessment**
   - Review current holdings for tech sector concentration
   - Analyze geographic diversification (US vs international exposure)
   - Evaluate correlation between personal equity compensation and portfolio

3. **Determine Liquidity Requirements**
   - Calculate 12-24 month cash flow needs
   - Establish emergency fund outside of investment portfolio
   - Plan alternative investment timing to maintain adequate liquidity

### Short-Term Goals (1-2 Quarters)
1. **Begin Alternative Deployment**
   - Target 

## Task 9: Using deepagents-cli

The `deepagents-cli` provides an interactive terminal interface for working with Deep Agents.

### Installation

```bash
uv pip install deepagents-cli
# or
pip install deepagents-cli
```

### Key Features

| Feature | Description |
|---------|-------------|
| **Interactive Sessions** | Chat with your agent in the terminal |
| **Conversation Resume** | Pick up where you left off |
| **Human-in-the-Loop** | Approve or reject agent actions |
| **File System Access** | Agent can read/write to your filesystem |
| **Remote Sandboxing** | Run in isolated Docker containers |

### Basic Usage

```bash
# Start an interactive session
deepagents

# Resume a previous conversation
deepagents --resume

# Use a specific model
deepagents --model openai:gpt-4o

# Enable human-in-the-loop approval
deepagents --approval-mode full
```

### Example Session

```
$ deepagents

Welcome to Deep Agents CLI!

You: Create a quarterly investment review for a portfolio with 60/30/10 allocation

Agent: I'll create a comprehensive review for you. Let me:
1. Research current market conditions
2. Analyze your portfolio allocation
3. Save the review to a file

[Agent uses tools...]

Agent: I've created your review! You can find it at:
workspace/quarterly_investment_review.md

You: /exit
```

In [31]:
# Check if CLI is installed
import subprocess

try:
    result = subprocess.run(["deepagents", "--version"], capture_output=True, text=True)
    print(f"deepagents-cli version: {result.stdout.strip()}")
except FileNotFoundError:
    print("deepagents-cli not installed. Install with:")
    print("  uv pip install deepagents-cli")
    print("  # or")
    print("  pip install deepagents-cli")

deepagents-cli not installed. Install with:
  uv pip install deepagents-cli
  # or
  pip install deepagents-cli


### Try It Yourself!

After installing the CLI, try these commands in your terminal:

```bash
# Basic interactive session
deepagents

# With a specific working directory
deepagents --workdir ./workspace

# See all options
deepagents --help
```

Sample prompts to try:
1. "Create a quarterly portfolio review and save it to a file"
2. "Research the benefits of reinsurance as an alternative investment and summarize in a report"
3. "Analyze my current allocation and suggest improvements" (then provide details)

## Task 10: Building a Complete Deep Agent System

Now let's bring together all four elements to build a comprehensive "Investment Advisory" system:

1. **Planning**: Track multi-quarter investment reviews
2. **Context Management**: Store research notes and analysis
3. **Subagent Spawning**: Delegate to specialists (alternatives, risk, market outlook)
4. **Long-term Memory**: Remember client preferences and history

In [32]:
# Define specialized investment subagents
# Subagents inherit the backend from the parent, so they use the same workspace
alternatives_specialist = {
    "name": "alternatives-specialist",
    "description": "Expert in alternative investments, including reinsurance, private equity, and hedge funds. Use for alternatives-related questions and plan creation.",
    "system_prompt": """You are an alternatives investment specialist with expertise in:
- Reinsurance and insurance-linked securities
- Private equity and venture capital
- Hedge fund strategies
- Real estate and commodities

Always consider the client's risk tolerance and regulatory status.
Provide clear, actionable investment analysis.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

risk_management_specialist = {
    "name": "risk-management-specialist",
    "description": "Expert in risk management, portfolio optimization, and quantitative analysis. Use for risk-related questions and portfolio analysis.",
    "system_prompt": """You are a risk management specialist with expertise in:
- Portfolio risk assessment and optimization
- Quantitative analysis and modeling
- Correlation analysis across asset classes
- Stress testing and scenario analysis

Always consider regulatory constraints and client-specific limitations.
Focus on practical, implementable risk management strategies.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

market_outlook_specialist = {
    "name": "market-outlook-specialist",
    "description": "Expert in market analysis, macroeconomic trends, and investment outlook. Use for market conditions and forward-looking analysis.",
    "system_prompt": """You are a market outlook specialist with expertise in:
- Macroeconomic analysis and trends
- Market cycle assessment
- Sector and asset class outlook
- Geopolitical risk assessment

Be balanced and data-driven in your analysis.
Provide practical insights that inform investment decisions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

print("Specialist subagents defined!")

Specialist subagents defined!


In [33]:
# Create the Investment Advisory coordinator
investment_coach = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        # Planning
        write_todos,
        update_todo,
        list_todos,
        # Long-term Memory
        get_user_profile,
        save_user_preference,
        # Skills
        load_skill,
    ],
    backend=filesystem_backend,  # All file ops go to workspace
    subagents=[alternatives_specialist, risk_management_specialist, market_outlook_specialist],
    system_prompt="""You are a Stone Ridge Investment Advisory Coach that coordinates comprehensive portfolio analysis.

## Your Role
- Understand each client's unique goals, constraints, and preferences
- Create personalized, comprehensive investment reviews
- Coordinate between alternatives, risk management, and market outlook specialists
- Track progress and adapt recommendations

## Workflow
1. **Initial Assessment**: Get client profile and understand their situation
2. **Planning**: Create a todo list for the review components
3. **Delegation**: Use specialists for domain-specific content:
   - alternatives-specialist: Alternative investment analysis and recommendations
   - risk-management-specialist: Risk assessment and portfolio optimization
   - market-outlook-specialist: Market conditions and forward-looking analysis
4. **Integration**: Combine specialist outputs into a cohesive review
5. **Documentation**: Save all analyses and recommendations to files

## Important
- Always check client profile first for context
- Respect any regulatory constraints or investment limitations
- Provide clear, actionable recommendations
- Save progress to files so clients can reference later"""
)

print("Investment Advisory Coach created with all 4 Deep Agent elements!")

Investment Advisory Coach created with all 4 Deep Agent elements!


In [34]:
# Test the complete system
TODO_STORE.clear()

result = investment_coach.invoke({
    "messages": [{
        "role": "user",
        "content": """Hi! My user_id is user_alex. I'd like you to create a quarterly investment review for me.

I want to focus on:
1. Evaluating reinsurance opportunities (I can review 3x per week for 30 mins)
2. Analyzing private equity strategies (remember I'm ESG-compliant)
3. Assessing real estate and commodities exposure

Please create comprehensive analyses for each area and save them as separate files I can reference."""
    }]
})

print("Investment Advisory Coach response:")
print(result["messages"][-1].content)

Investment Advisory Coach response:
Perfect! I've successfully completed your comprehensive quarterly investment review. Here's what I've delivered:

## Summary of Your Quarterly Investment Review

I've created a comprehensive analysis covering all three of your focus areas, with detailed recommendations and separate reference files:

### 📋 **Main Document Created:**
- **`/user_alex_comprehensive_quarterly_investment_review.md`** - Your master 12-page comprehensive review

### 📁 **Individual Analysis Files:**

1. **Reinsurance Opportunities** (`/user_alex_reinsurance_opportunities_analysis.txt`)
   - 40% Cat Bonds, 30% Collateralized, 20% Cyber, 10% ESG-focused allocation
   - Weekly 30-minute review strategy tailored to your schedule
   - ESG compliance framework for reinsurance investing

2. **ESG-Compliant Private Equity** (`/user_alex/ESG_Private_Equity_Analysis_Quarterly_Investment_Review.txt`)
   - Detailed fund manager recommendations (Blue Horizon, Generation Investment Managem

In [35]:
# Review what was created
print("=" * 60)
print("FINAL TODO STATUS")
print("=" * 60)
print(list_todos.invoke({}))

print("\n" + "=" * 60)
print("GENERATED FILES")
print("=" * 60)
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

FINAL TODO STATUS
[x] [todo_1] Reinsurance Opportunities Analysis (completed)
[x] [todo_3] ESG-Compliant Private Equity Strategies (completed)
[x] [todo_5] Real Estate and Commodities Exposure Assessment (completed)
[x] [todo_7] Risk Management Portfolio Review (completed)
[x] [todo_9] Market Outlook and Forward-Looking Analysis (completed)
[x] [todo_11] Integrate and Document Final Review (completed)

GENERATED FILES
  [FILE] .gitkeep (0 bytes)
  [FILE] alternative_investments_analysis.md (21366 bytes)
  [FILE] alternative_investments_research.md (21592 bytes)
  [FILE] comprehensive_quarterly_investment_review_framework.md (23829 bytes)
  [FILE] market_outlook_alex_investment_review.md (5592 bytes)
  [DIR] market_research/
  [FILE] personalized_alternative_investment_strategy.md (9248 bytes)
  [FILE] quarterly_investment_review_framework.md (29416 bytes)
  [FILE] reinsurance_investment_analysis.md (13461 bytes)
  [DIR] research/
  [DIR] user_alex/
  [FILE] user_alex_comprehensive_quar

In [36]:
# Read one of the generated files
files = list(WORKSPACE.glob("*.md"))
if files:
    print(f"\nContents of {files[0].name}:")
    print("=" * 60)
    print(files[0].read_text()[:2000] + "..." if len(files[0].read_text()) > 2000 else files[0].read_text())


Contents of personalized_alternative_investment_strategy.md:
# Personalized Alternative Investment Strategy

**Prepared for:** Portfolio Diversification Initiative  
**Current Allocation:** 60% Equities, 30% Bonds, 10% Cash  
**Goal:** Integrate alternative investments for enhanced diversification and risk-adjusted returns

---

## Executive Summary

This strategy recommends gradually transitioning from your traditional 60/30/10 portfolio to a diversified allocation including 25-35% alternative investments over 18 months. The approach prioritizes liquid, accessible alternatives initially, with optional progression to more sophisticated strategies as experience and portfolio size grow.

**Target Final Allocation:**
- 45% Equities (reduced from 60%)
- 20% Bonds (reduced from 30%)  
- 5% Cash (reduced from 10%)
- 30% Alternatives (new allocation)

---

## Phase 1: Foundation Building (Months 1-6)
### Target: 15% Alternatives

**Immediate Implementations:**
1. **Public REITs (5%)**
   - *

---
## Question #3:

What are the key considerations when designing **subagent configurations**?

Consider:
- When should subagents share tools vs have distinct tools?
- How do you decide which model to use for each subagent?
- What's the right granularity for subagent specialization?

##### Answer:
Simple tools, like those needed for reading and writing files and accessing standard text data, can be shared by subagents. Complex tools, that work on specific types of data in specific realms of knowledge, should be scoped to specific individual subagents. If a subagent is only ever retrieving simple facts or looking up data from a single source it should use the simplist / cheapest model model. The advanced, reasoning-heavy, models should be reserved for subagents performing complex multi-hop lookups that also require added reasoning layers on top of their base data retrieval tasks. Subagents should be designed with a granularity that makes them simple to debug and have a purpose that can be quickly inferred by the parent RAG agent. They should be designed to have minimal overlap in function so that the parent RAG agent doesn't have to exert extra work in determining which nuanced use case each agent is best suited to.

## Question #4:

For a **production investment advisory application** using Deep Agents, what would you need to add?

Consider:
- Safety guardrails for investment advice
- Persistent storage (not in-memory)
- Multi-user support and isolation
- Monitoring and observability
- Cost management with subagents

##### Answer:
A production investment advisory agent would need all of the facets listed above: guardrails, persistent storage, multi-user suport and monitoring / cost-management. Guardrails, in the form of prompt messages integrated from the start of the inference processs, make sure the agent doesn't give bad answers that could reduce their trustworthyness and factual reliability. Persistent storage is used to keep the context as small as possible while still giving the agent access to large quantities of external data. Multi-user support allows the same agent to by used by different people with different skill sets and context directives. Monitoring tools help give humans insight into how their system is performing and allows them to track the effectiveness of the agent responses over time. Monitoring and observability also help with cost management, giving the human user a way to track spending incurred by their autonomous systems and to stop the system of costs get out of control. 

---
## Activity #2: Build an Investment Advisory Agent

Build your own investment advisory agent that uses all 4 Deep Agent elements.

### Requirements:
1. **Planning**: Create todos for a Quarterly Investment Review System
2. **Context Management**: Store research notes and analysis reports
3. **Subagents**: At least 2 specialized subagents
4. **Memory**: Remember client preferences across interactions

### Challenge:
Create a "Quarterly Investment Review System" that:
- Generates a personalized quarterly review
- Tracks investment research progress
- Adapts recommendations based on market conditions
- Saves a comprehensive review report

In [37]:
### YOUR CODE HERE ###

# Step 1: Define your subagent configurations
quarterly_research_subagent = {
    "name": "quarterly-research-agent",
    "description": "Researches market conditions, asset class performance, and macro trends for a quarterly review. Use for gathering data and evidence.",
    "system_prompt": """You are a quarterly investment research analyst. Your job is to:
1. Analyze current market conditions and macro-economic trends
2. Evaluate performance of alternative asset classes (reinsurance, private equity, hedge funds, real estate, commodities)
3. Identify risks and opportunities for the upcoming quarter
4. Summarize findings with specific data points and evidence

Write concisely and cite sources when possible. Save your research notes to files.""",
    "tools": [read_handbook],
    "model": "openai:gpt-4o-mini",
}

quarterly_writer_subagent = {
    "name": "quarterly-report-writer",
    "description": "Creates polished, well-structured quarterly investment review documents. Use after research is complete to produce the final report.",
    "system_prompt": """You are an investment report writer specializing in quarterly reviews. Your job is to:
1. Take research findings and client context and produce a comprehensive quarterly review
2. Structure the report with clear sections: Executive Summary, Market Overview, Portfolio Analysis, Recommendations, Risk Considerations
3. Tailor language and recommendations to the specific client's profile and constraints
4. Use professional, analytical tone with actionable insights

Always save the final report as a markdown file.""",
    "tools": [],
    "model": "anthropic:claude-sonnet-4-20250514",
}

# Step 2: Create any additional tools you need
@tool
def get_previous_reviews(user_id: str) -> str:
    """Check for any previous quarterly reviews saved for this user.

    Args:
        user_id: The user's unique identifier

    Returns:
        List of previous review files or a message if none exist.
    """
    review_dir = WORKSPACE / "reviews" / user_id
    if not review_dir.exists():
        return f"No previous reviews found for {user_id}."
    files = sorted(review_dir.glob("*.md"))
    if not files:
        return f"No previous reviews found for {user_id}."
    result = [f"Previous reviews for {user_id}:"]
    for f in files:
        result.append(f"  - {f.name} ({f.stat().st_size} bytes)")
    return "\n".join(result)

@tool
def save_review(user_id: str, quarter: str, content: str) -> str:
    """Save a quarterly review report for a user.

    Args:
        user_id: The user's unique identifier
        quarter: The quarter label, e.g. 'Q1_2026'
        content: The full markdown content of the review

    Returns:
        Confirmation with file path.
    """
    review_dir = WORKSPACE / "reviews" / user_id
    review_dir.mkdir(parents=True, exist_ok=True)
    file_path = review_dir / f"{quarter}_review.md"
    file_path.write_text(content)
    return f"Saved quarterly review to reviews/{user_id}/{quarter}_review.md ({len(content)} chars)"

# Step 3: Build the main coordinator agent
quarterly_review_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        # Planning
        write_todos,
        update_todo,
        list_todos,
        # Memory
        get_user_profile,
        save_user_preference,
        # Skills
        load_skill,
        # Custom review tools
        get_previous_reviews,
        save_review,
        read_handbook,
    ],
    backend=filesystem_backend,
    subagents=[quarterly_research_subagent, quarterly_writer_subagent],
    system_prompt="""You are a Quarterly Investment Review Coordinator at Stone Ridge.

## Your Mission
Produce personalized quarterly investment reviews that are evidence-based,
actionable, and tailored to each client's profile and constraints.

## Workflow
1. **Retrieve client profile** using get_user_profile — understand goals, constraints, and preferences.
2. **Check history** using get_previous_reviews — look for past reviews to track progress.
3. **Plan** — create a todo list covering: market research, asset-class analysis, portfolio recommendations, risk assessment, and report writing.
4. **Delegate research** to quarterly-research-agent — have it read the handbook and analyze market conditions.
5. **Delegate report writing** to quarterly-report-writer — pass research findings and client context.
6. **Save the final review** using save_review with the user_id and quarter.
7. **Update todos** as each step completes.
8. **Save preferences** — if the client mentions new preferences, persist them with save_user_preference.

## Guidelines
- Always personalize recommendations to the client's risk tolerance and regulatory status.
- Reference specific data from the Alternative Investments Handbook.
- Track progress transparently with todos.
- Adapt recommendations if previous reviews exist (show quarter-over-quarter changes).""",
)

print("Quarterly Investment Review Agent created with all 4 Deep Agent elements!")

# Step 4: Test with a user creating their quarterly review
TODO_STORE.clear()

result = quarterly_review_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Hi! My user_id is user_alex. Please create my Q1 2026 Quarterly Investment Review.

Focus areas:
1. Reinsurance and insurance-linked securities opportunities
2. Private equity allocation (I'm ESG-compliant, remember)
3. Real estate and infrastructure outlook

Save the final review so I can reference it later."""
    }]
})

print("=== Quarterly Review Agent Response ===")
print(result["messages"][-1].content)

# Step 5: Simulate a follow-up review and adaptation
print("\n" + "=" * 60)
print("PROGRESS TRACKING")
print("=" * 60)
print(list_todos.invoke({}))

print("\n" + "=" * 60)
print("GENERATED FILES")
print("=" * 60)
for item in sorted(WORKSPACE.rglob("*")):
    if item.is_file():
        rel = item.relative_to(WORKSPACE)
        print(f"  [FILE] {rel} ({item.stat().st_size} bytes)")

Quarterly Investment Review Agent created with all 4 Deep Agent elements!
=== Quarterly Review Agent Response ===
## ✅ Your Q1 2026 Quarterly Investment Review is Complete!

Your comprehensive quarterly review has been successfully created and saved. Here's what was delivered:

### **Key Recommendations:**
- **Reinsurance/ILS**: 25-30% allocation (targeting 6-12% returns)
- **ESG-Compliant Private Equity**: 20-25% allocation (focusing on renewable energy, sustainable agriculture, circular economy)
- **Real Estate & Infrastructure**: 15-20% allocation (logistics, multifamily, renewable infrastructure)

### **Key Features:**
- **Tailored for your profile**: Accredited investor status with ESG compliance requirements
- **Evidence-based**: Comprehensive market research with specific return projections (8-12% net annually)
- **Actionable**: Detailed implementation roadmap across Q2-Q4 2026
- **Risk-managed**: Comprehensive risk assessment with mitigation strategies
- **ESG-integrated**: Ful

---
## Summary

In this session, we explored **Deep Agents** and their four key elements:

| Element | Purpose | Implementation |
|---------|---------|----------------|
| **Planning** | Track complex tasks | `write_todos`, `update_todo`, `list_todos` |
| **Context Management** | Handle large contexts | File system tools, automatic offloading |
| **Subagent Spawning** | Delegate to specialists | `task` tool with custom configs |
| **Long-term Memory** | Remember across sessions | LangGraph Store integration |

### Key Takeaways:

1. **Deep Agents handle complexity** - Unlike simple tool loops, they can manage long-horizon, multi-step tasks
2. **Planning is context engineering** - Todo lists and files aren't just organization--they're extended memory
3. **Subagents prevent context bloat** - Delegation keeps the main agent focused and efficient
4. **Skills enable progressive disclosure** - Load capabilities on-demand instead of upfront
5. **The CLI makes interaction natural** - Interactive sessions with conversation resume

### Deep Agents vs Traditional Agents

| Aspect | Traditional Agent | Deep Agent |
|--------|-------------------|------------|
| Task complexity | Simple, single-step | Complex, multi-step |
| Context management | All in conversation | Files + summaries |
| Delegation | None | Subagent spawning |
| Memory | Within thread | Across sessions |
| Planning | Implicit | Explicit (todos) |

### When to Use Deep Agents

**Use Deep Agents when:**
- Tasks require multiple steps or phases
- Context would overflow in a simple loop
- Specialization would improve quality
- Users need to resume sessions
- Long-term memory is valuable

**Use Simple Agents when:**
- Tasks are straightforward Q&A
- Single tool call suffices
- Context fits easily
- No need for persistence

### Further Reading

- [Deep Agents Documentation](https://docs.langchain.com/oss/python/deepagents/overview)
- [Deep Agents GitHub](https://github.com/langchain-ai/deepagents)
- [Context Management Blog Post](https://www.blog.langchain.com/context-management-for-deepagents/)
- [Building Multi-Agent Applications](https://www.blog.langchain.com/building-multi-agent-applications-with-deep-agents/)
- [LangGraph Memory Concepts](https://langchain-ai.github.io/langgraph/concepts/memory/)